In [1]:
# Setup variables
BRANCH = "main"
import os
os.environ['MLFLOW_TRACKING_URI'] = "http://100.101.196.27:5000"
os.environ['COLAB_GPU'] = "True"


In [ ]:
# Mount google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Clone repo
if not os.path.exists("/content/inat-phenology-cv"):
    !git clone -b {BRANCH} https://github.com/etiennegodin/inat-phenology-cv.git /content/inat-phenology-cv
%cd /content/inat-phenology-cv
!git pull origin {BRANCH} -q
!git restore .

Cloning into '/content/inat-phenology-cv'...
remote: Enumerating objects: 1514, done.
remote: Counting objects: 100% (669/669), done.
remote: Compressing objects: 100% (288/288), done.
remote: Total 1514 (delta 428), reused 547 (delta 312), pack-reused 845 (from 1)
Receiving objects: 100% (1514/1514), 263.15 KiB | 10.12 MiB/s, done.
Resolving deltas: 100% (953/953), done.
/content/inat-phenology-cv


In [ ]:
%%capture
# Instal project requirements
!pip install -q --upgrade pip
!pip install -e . -q


In [ ]:
%%capture
#Setup colab network to mlflow server
!curl -fsSL https://tailscale.com/install.sh | sh
!pip install -q "requests[socks]"

In [ ]:
%%bash
sudo setsid nohup bash -c '
while true; do
  tailscaled \
    --tun=userspace-networking \
    --socks5-server=localhost:1055 \
    --state=/tmp/tailscale.state \
    >> /tmp/tailscaled.log 2>&1
  echo "[$(date)] tailscaled exited, restarting in 2s" >> /tmp/tailscaled.log
  sleep 2
done
' < /dev/null > /tmp/tailscaled_supervisor.log 2>&1 &
echo "supervisor launched"

supervisor launched


In [ ]:
import subprocess
from google.colab import userdata
ts_auth_key = userdata.get("TAILSCALE_AUTH_KEY")
assert ts_auth_key
subprocess.run(
    [
        "sudo",
        "tailscale",
        "up",
        "--auth-key",
        ts_auth_key,
    ],
    check=True,
)
del ts_auth_key

In [ ]:
!sudo tailscale status

100.79.154.107   5e9ea76b8795                       manateetiti@  linux    -                           
100.112.144.117  c318ba5607df                       manateetiti@  linux    offline, last seen 36m ago  
100.101.196.27   etienne-lenovo-ideapad-flex-15iml  manateetiti@  linux    -                           
100.112.113.49   pixel-7-pro                        manateetiti@  android  offline, last seen 18d ago  


In [ ]:
import requests

mlflow_proxies = {
    "http": "socks5h://localhost:1055",
    "https": "socks5h://localhost:1055",
}

r = requests.get(
    "http://100.101.196.27:5000/version",
    proxies=mlflow_proxies,
    timeout=5,
)

print(r.status_code)
print(r.text)

200
3.15.1


In [ ]:
# Copying mlflow.db from drive
os.makedirs("/content/data", exist_ok=True)
!cp -r "/content/drive/MyDrive/inat-phenology-cv/data/cv_photos3.parquet" "/content/data/"

In [ ]:
print("Copying images to local disk...")
if not os.path.exists("/content/images"):
  os.makedirs("/content/images", exist_ok=True)
  !tar -xf "/content/drive/MyDrive/inat-phenology-cv/data/images.tar.gz" -C /content/
  os.environ["INAT_IMAGE_DIR"] = "/content/images"

Copying images to local disk...


In [ ]:
!nvidia-smi

Sat Sep  5 19:44:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Gated attention, no dropout
!torch_pipe train -n 30 --backbone bioclip --gated --attention_dropout 0.0 --seed 42 -name 'cv_inat_bioclip' --unfreeze 2 -lr 0.00005


INFO: Starting
Connecting to mlflow
Initalizing experiment
Running on cuda
open_clip_config.json: 100% 469/469 [00:00<00:00, 1.90MB/s]

open_clip_pytorch_model.bin: downloading bytes:  37% 220M/599M [00:02<00:01, 229MB/s, 15.1MB/s  ]
open_clip_pytorch_model.bin: downloading bytes:  48% 289M/599M [00:02<00:01, 264MB/s, 23.8MB/s  ]
open_clip_pytorch_model.bin: downloading bytes:  67% 404M/599M [00:02<00:00, 357MB/s, 30.6MB/s  ]
open_clip_pytorch_model.bin: downloading bytes:  78% 466M/599M [00:02<00:00, 405MB/s, 37.2MB/s  ]
open_clip_pytorch_model.bin: downloading bytes:  89% 530M/599M [00:02<00:00, 376MB/s, 47.1MB/s  ]
open_clip_pytorch_model.bin: reconstructing file:  90% 537M/599M [00:02<00:00, 303MB/s, 43.2MB/s  ]
open_clip_pytorch_model.bin: downloading bytes: 100% 567M/567M [00:03<00:00, 181MB/s, 49.5MB/s  ]
open_clip_pytorch_model.bin: reconstructing file: 100% 599M/599M [00:03<00:00, 191MB/s, 53.4MB/s  ]
Loading 14228 images into RAM... This may take a while...
100% 14228/14228 [

In [ ]:
from google.colab import runtime
runtime.unassign()